# Practical Homework 1 — Audio Enhancement & Image-Based Fracture Detection

### Moeein Aali - 401105561

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import librosa
import librosa.display
import soundfile as sf
from scipy.signal import (butter, sosfiltfilt, sosfreqz, iirnotch, tf2sos,
                          find_peaks, lfilter)
from scipy.ndimage import median_filter, binary_dilation, binary_closing, uniform_filter1d
import cv2
from skimage.filters import apply_hysteresis_threshold, frangi
from skimage.measure import label, regionprops

plt.rcParams["figure.dpi"] = 90
plt.rcParams["axes.grid"] = False

DATA_DIR = Path(".")
OUT_DIR = Path("outputs")
OUT_DIR.mkdir(exist_ok=True)

AUDIO_FILES = ["2.mp3", "3.mp3", "4.mp3"]

# STFT frame parameters used consistently for all audio analysis
FRAME, HOP = 2048, 512

# Problem 1 — Call-Center Audio Restoration & Pre-Processing

## Background theory (used throughout Problem 1)

**Digital audio.** A microphone measures sound pressure as a continuous signal $x(t)$; the
recording stores samples $x[n] = x(n/f_s)$ taken $f_s$ times per second. Our files use
$f_s = 44100\,$Hz, so by the Nyquist–Shannon sampling theorem they can represent frequencies up
to $f_s/2 = 22.05\,$kHz. Human speech carries almost all of its intelligibility between roughly
**300 Hz and 3.4 kHz** — which is exactly the band a classic telephone channel transmits.

**Decibels.** Amplitude ratios are reported in dB: $20\log_{10}(a/a_{ref})$. We use dBFS
(full-scale = 1.0). Every 20 dB is a 10× amplitude ratio.

**Short-Time Fourier Transform (STFT).** Speech is *non-stationary*, so a single Fourier
transform of the whole file hides *when* things happen. The STFT slices the signal into short
overlapping windows (here 2048 samples ≈ 46 ms, hop 512 ≈ 12 ms), applies a window function and
an FFT to each slice, and yields $X[k, m]$ — the content of frequency bin $k$ around frame $m$.
Its magnitude displayed as an image is the **spectrogram**: the single most useful diagnostic
tool in this problem, because each artifact has a distinct spectrographic signature:

| Artifact | Signature in the spectrogram |
|---|---|
| Band-limited channel | energy abruptly cut above ~3.4 kHz |
| Stationary noise (hiss/hum) | a constant "floor" / horizontal lines at fixed frequencies |
| Transients (clicks, crackle) | thin full-band vertical stripes |
| Hold music | long-sustained horizontal harmonic lines |
| Silence / dead air | black columns |
| Echo | ghost copies of speech patterns shifted in time |

## 1.1 Exploratory Data Analysis (EDA)

For each recording we plot the waveform, the spectrogram (0–8 kHz so the band-limit is visible)
and the short-time RMS energy in dB. The quantitative summary below each figure measures:
duration, peak, RMS level, the fraction of frames more than 40 dB below the loudest frame
("silence fraction"), and the fraction of spectral energy above 3.4 kHz (a band-limitation
indicator).

In [ ]:
def show_table(headers, rows, title=None):
    """Print a small aligned text table (avoids extra dependencies)."""
    if title:
        print(title)
    widths = [max(len(str(h)), *(len(str(r[i])) for r in rows)) for i, h in enumerate(headers)]
    line = "  ".join(str(h).ljust(w) for h, w in zip(headers, widths))
    print(line)
    print("-" * len(line))
    for r in rows:
        print("  ".join(str(v).ljust(w) for v, w in zip(r, widths)))

def frame_rms_db(y):
    rms = librosa.feature.rms(y=y, frame_length=FRAME, hop_length=HOP)[0]
    return 20 * np.log10(rms + 1e-10)


def plot_overview(y, sr, name):
    fig, axes = plt.subplots(3, 1, figsize=(14, 8))
    t = np.arange(len(y)) / sr
    axes[0].plot(t[::20], y[::20], lw=0.3)
    axes[0].set_xlim(0, t[-1]); axes[0].set_ylabel("amplitude")
    axes[0].set_title(f"{name} — waveform")

    S_db = librosa.amplitude_to_db(np.abs(librosa.stft(y, n_fft=FRAME, hop_length=HOP)), ref=np.max)
    librosa.display.specshow(S_db, sr=sr, hop_length=HOP, x_axis="time", y_axis="hz",
                             ax=axes[1], cmap="magma")
    axes[1].set_ylim(0, 8000)
    axes[1].set_title(f"{name} — spectrogram (0–8 kHz)")

    db = frame_rms_db(y)
    tf = np.arange(len(db)) * HOP / sr
    axes[2].plot(tf, db, lw=0.6)
    axes[2].set_xlim(0, t[-1]); axes[2].set_ylabel("frame RMS (dB)")
    axes[2].set_xlabel("time (s)")
    axes[2].set_title(f"{name} — short-time energy")
    fig.tight_layout()
    plt.show()


audio_raw = {}
eda_rows = []
for name in AUDIO_FILES:
    y, sr = librosa.load(DATA_DIR / name, sr=None, mono=True)
    audio_raw[name] = (y, sr)
    plot_overview(y, sr, name)

    db = frame_rms_db(y)
    S = np.abs(librosa.stft(y, n_fft=4096))
    freqs = librosa.fft_frequencies(sr=sr, n_fft=4096)
    P = S ** 2                                   # power, so the ratio is true energy
    hi_energy = P[freqs >= 3400].sum() / P.sum()
    eda_rows.append([
        name, f"{len(y)/sr:.1f}", sr, f"{np.max(np.abs(y)):.2f}",
        f"{20*np.log10(np.sqrt(np.mean(y**2))):.1f}",
        f"{100*np.mean(db < db.max() - 40):.0f}%",
        f"{100*hi_energy:.2f}%",
    ])

show_table(
    ["file", "dur (s)", "sr", "peak", "RMS dBFS", "silence@-40dB", "energy>3.4kHz"],
    eda_rows, "Summary of the three recordings:")

----

### EDA findings — where each artifact lives

* **`2.mp3` (214 s).** Energy is confined below ≈3 kHz (essentially 0 % above 3.4 kHz): a classic
  telephone-band recording. The energy track shows a long **dead-air block around t ≈ 68–102 s**
  (~ -58 dB, with a few faint blips) plus many shorter pauses — ideal for VAD. The noise floor
  between words is clearly audible hiss.
* **`3.mp3` (188 s).** Loud (decoded peak > 1 → the channel was driven into clipping) and almost
  continuous (~1 % silence). Content is **broadband** (~5 % of signal energy above 3.4 kHz,
  versus essentially zero in the other two files), with many
  thin full-band vertical stripes — impulsive crackle. Between **t ≈ 90–135 s** the spectrogram
  shows long sustained tonal lines: **hold music / hold tone** (verified quantitatively in §1.6).
* **`4.mp3` (154 s).** Telephone-band like `2.mp3` (~0.4 % above 3.4 kHz), silences spread through
  the file, a hard click near t ≈ 92 s and a few near the end.

The next cell looks for **persistent narrowband components** (stationary hum and sustained
tones). Speech is non-stationary, so the *median* magnitude spectrum over time suppresses it and
exposes anything constant. All three files carry a strong mains-hum line at ~48–50 Hz, and
`2.mp3`/`4.mp3` show a related low-order harmonic near 242–253 Hz (≈ 5 × 50 Hz, the frequency
readings are quantized to ~5.4 Hz FFT bins) — all below the 300 Hz band edge, so the band-pass
will remove them.
More interesting are the *in-band* persistent lines: **2966 Hz in `4.mp3`** and **2444 Hz in
`3.mp3`** — narrowband interference sitting right inside the speech band, which motivates the
per-file auto-notching stage in §1.2. (Inside `3.mp3`'s hold-music region there are additional
sustained tones near 2143/2245 Hz — analyzed separately in §1.6.)

In [ ]:
def persistent_spectrum(y, sr, n_fft=8192):
    """Median magnitude spectrum over time (dB) — reveals stationary components."""
    S = np.abs(librosa.stft(y, n_fft=n_fft, hop_length=2048))
    freqs = librosa.fft_frequencies(sr=sr, n_fft=n_fft)
    return freqs, 20 * np.log10(np.median(S, axis=1) + 1e-10)


def persistent_peaks(y, sr, fmin=30, fmax=4000, prom_db=8):
    freqs, med_db = persistent_spectrum(y, sr)
    m = (freqs >= fmin) & (freqs <= fmax)
    pk, props = find_peaks(med_db[m], prominence=prom_db, distance=15)
    return [(freqs[m][i], p) for i, p in zip(pk, props["prominences"])]


fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=True)
for ax, name in zip(axes, AUDIO_FILES):
    y, sr = audio_raw[name]
    freqs, med_db = persistent_spectrum(y, sr)
    m = freqs < 4000
    ax.plot(freqs[m], med_db[m], lw=0.7)
    for f0, p in persistent_peaks(y, sr):
        ax.annotate(f"{f0:.0f}", (f0, med_db[np.argmin(np.abs(freqs - f0))]),
                    fontsize=7, color="red")
    ax.set_ylabel("median |X| (dB)")
    ax.set_title(f"{name} — persistent (stationary) spectrum, peaks annotated in Hz")
axes[-1].set_xlabel("frequency (Hz)")
fig.tight_layout(); plt.show()

for name in AUDIO_FILES:
    y, sr = audio_raw[name]
    pks = persistent_peaks(y, sr)
    print(f"{name}: persistent peaks -> " +
          ", ".join(f"{f:.0f} Hz (+{p:.0f} dB)" for f, p in pks[:10]))

## 1.2 Enhancement building blocks

The pipeline is built from small, testable functions. Order of application (and why):

1. **Peak pre-normalization** — undo decoder overshoot (`3.mp3` decodes to peak 2.03) so all
   later thresholds work on a comparable scale.
2. **Band-pass 300–3400 Hz (Butterworth, zero-phase)** — remove everything a telephone channel
   never carried: rumble + mains hum below 300 Hz, hiss and broadband junk above 3.4 kHz.
   A **Butterworth** design is chosen because its passband is *maximally flat* (no ripple →
   no coloration of speech); we apply it with `sosfiltfilt`, i.e. forwards *and* backwards, which
   squares the magnitude response and cancels the phase response (**zero-phase**: no waveform
   smearing, no group-delay distortion) and use second-order sections (SOS) for numerical
   stability.
3. **Auto-detected notch filters for in-band persistent tones** — the strong mains-hum series
   (50/150/250 Hz) dies below the 300 Hz band edge, but each file may carry its own narrowband
   interference *inside* the passband (e.g. a persistent 2966 Hz line in `4.mp3`). We reuse the
   median-spectrum analysis from the EDA: any peak in 310–3390 Hz with ≥10 dB prominence in the
   *median* (i.e. stationary) spectrum is interference, never speech — speech formants move and
   average out over minutes of audio. Each detected tone is removed with an IIR notch (very
   narrow band-stop; per pass the −3 dB width is f₀/Q = f₀/35, and the forward–backward
   `filtfilt` application deepens the null) that leaves neighbouring speech untouched.
4. **Adaptive transient limiter (de-click)** — clicks/crackle are samples that tower above the
   *local* signal level. We compute a 200 ms rolling RMS and softly compress any sample that
   exceeds 5× the local RMS. A *global* threshold would fail here: loud speech in a quiet file
   would be clipped, and quiet-section clicks would pass.
5. **Spectral subtraction** — removes the *stationary* noise floor (hiss). See §1.4.
6. **Hold-music suppression** (only where detected) — §1.6.
7. **Echo analysis** — §1.7.
8. **VAD + silence stripping** — §1.5.
9. **RMS loudness normalization to −20 dBFS** — a call-center corpus should have uniform
   loudness; RMS (not peak) normalization equalizes *perceived* level across files and speakers.

In [ ]:
def peak_normalize(y, peak=0.95):
    return y * (peak / (np.max(np.abs(y)) + 1e-12))


def bandpass_filter(y, sr, lo=300.0, hi=3400.0, order=4):
    """Zero-phase Butterworth band-pass (applied via second-order sections)."""
    sos = butter(order, [lo, hi], btype="bandpass", fs=sr, output="sos")
    return sosfiltfilt(sos, y)


def notch_filter(y, sr, freqs, q=35.0):
    """Cascade of narrow IIR notches at the given frequencies."""
    for f0 in freqs:
        b, a = iirnotch(f0, q, fs=sr)
        y = sosfiltfilt(tf2sos(b, a), y)
    return y


def adaptive_declick(y, sr, k=5.0, win_s=0.2, soft=0.15, abs_floor=0.02):
    """Softly limit samples exceeding k x local RMS (200 ms window)."""
    local_rms = np.sqrt(uniform_filter1d(y ** 2, size=int(win_s * sr)))
    thr = np.maximum(k * local_rms, abs_floor)
    over = np.abs(y) > thr
    out = y.copy()
    out[over] = np.sign(y[over]) * (thr[over] + soft * (np.abs(y[over]) - thr[over]))
    return out, int(over.sum())


def rms_normalize(y, target_db=-20.0):
    gain = 10 ** (target_db / 20) / (np.sqrt(np.mean(y ** 2)) + 1e-12)
    out = y * gain
    peak = np.max(np.abs(out))
    if peak > 0.99:          # safety: never clip
        out *= 0.99 / peak
    return out


def inband_persistent_tones(y, sr, fmin=310.0, fmax=3390.0, prom_db=10.0):
    """Frequencies of stationary narrowband interference inside the speech band."""
    return [f for f, _ in persistent_peaks(y, sr, fmin=fmin, fmax=fmax, prom_db=prom_db)]


# Detect each file's in-band interference tones once (used by the full pipeline later)
PERSISTENT_TONES = {}
for name in AUDIO_FILES:
    y, sr = audio_raw[name]
    PERSISTENT_TONES[name] = inband_persistent_tones(peak_normalize(y), sr)
    print(f"{name}: in-band persistent tones to notch -> "
          + (", ".join(f"{f:.0f} Hz" for f in PERSISTENT_TONES[name]) or "none"))

# --- Frequency responses of the filters, as designed ---
sos_bp = butter(4, [300, 3400], btype="bandpass", fs=44100, output="sos")
w, h = sosfreqz(sos_bp, worN=8192, fs=44100)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].semilogx(w, 20 * np.log10(np.abs(h) + 1e-12))
axes[0].axvline(300, color="r", ls="--", lw=0.8); axes[0].axvline(3400, color="r", ls="--", lw=0.8)
axes[0].set_ylim(-80, 5); axes[0].set_xlim(20, 22050)
axes[0].set_title("Butterworth band-pass 300–3400 Hz (order 4, single pass)")
axes[0].set_xlabel("Hz"); axes[0].set_ylabel("gain (dB)"); axes[0].grid(True, which="both", alpha=.3)

demo_tones = PERSISTENT_TONES["4.mp3"] or [1000.0]
for f0 in demo_tones:
    b, a = iirnotch(f0, 35, fs=44100)
    wn, hn = sosfreqz(tf2sos(b, a), worN=65536, fs=44100)
    axes[1].plot(wn, 20 * np.log10(np.abs(hn) + 1e-12), label=f"notch {f0:.0f} Hz (Q=35)")
axes[1].set_xlim(min(demo_tones) - 400, max(demo_tones) + 400); axes[1].set_ylim(-50, 3)
axes[1].set_title("IIR notch response: surgical, -3 dB width = f0/Q")
axes[1].set_xlabel("Hz"); axes[1].legend(); axes[1].grid(True, alpha=.3)
fig.tight_layout(); plt.show()

## 1.3 Band-pass in action

Below: 12 s of `2.mp3` before and after the band-pass (plus any detected tone notches). The
high-frequency hiss floor and the sub-300 Hz rumble (with its hum lines at 48/242 Hz) disappear;
the speech band is untouched (Butterworth passband is flat, filtfilt is zero-phase).

In [ ]:
y2, sr2 = audio_raw["2.mp3"]
seg = peak_normalize(y2)[int(120 * sr2):int(132 * sr2)]
seg_bp = notch_filter(bandpass_filter(seg, sr2), sr2, PERSISTENT_TONES["2.mp3"])

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, (s, title) in zip(axes, [(seg, "before"), (seg_bp, "after band-pass + notches")]):
    S_db = librosa.amplitude_to_db(np.abs(librosa.stft(s, n_fft=FRAME, hop_length=HOP)), ref=np.max)
    librosa.display.specshow(S_db, sr=sr2, hop_length=HOP, x_axis="time", y_axis="hz",
                             ax=ax, cmap="magma", vmin=-80)
    ax.set_ylim(0, 8000); ax.set_title(f"2.mp3 (t=120–132 s) — {title}")
fig.tight_layout(); plt.show()

---

## 1.4 Stationary-noise removal: spectral subtraction

**Idea.** Stationary noise (hiss) has a magnitude spectrum that barely changes over time. If we
can estimate the noise magnitude $|\hat N[k]|$, we can subtract it from every STFT frame:

$$|\hat X_{clean}[k,m]| = \max\big(|X[k,m]| - \alpha\,|\hat N[k]|,\; \beta\,|\hat N[k]|\big)$$

and resynthesize with the **original phase** (the ear is largely insensitive to phase here).

* The noise estimate is the average magnitude of the **5 % lowest-energy frames** — an automatic
  way of picking "silent" frames, which works because every file contains speech pauses.
* $\alpha = 2$ (**over-subtraction**) compensates for the fact that noise fluctuates around its
  average — subtracting exactly the average would leave half the noise behind.
* $\beta = 0.05$ (**spectral floor**) prevents negative/zero bins. Without a floor, bins that
  fluctuate below zero get clamped in isolation and turn into randomly appearing/disappearing
  tone burst artifacts — the well-known **"musical noise"** of spectral subtraction. Keeping a
  small constant floor masks this.

In [ ]:
def spectral_subtraction(y, sr, alpha=2.0, beta=0.05, quiet_frac=0.05):
    """Magnitude spectral subtraction with over-subtraction and a spectral floor."""
    S = librosa.stft(y, n_fft=FRAME, hop_length=HOP)
    mag, phase = np.abs(S), np.angle(S)
    frame_energy = np.sqrt((mag ** 2).mean(axis=0))
    n_quiet = max(10, int(quiet_frac * len(frame_energy)))
    quiet_idx = np.argsort(frame_energy)[:n_quiet]        # the quietest frames = noise sample
    noise_mag = mag[:, quiet_idx].mean(axis=1, keepdims=True)
    clean_mag = np.maximum(mag - alpha * noise_mag, beta * noise_mag)
    return librosa.istft(clean_mag * np.exp(1j * phase), hop_length=HOP, length=len(y)), noise_mag


# Demonstration on a chunk of 2.mp3 that contains both dead air and speech (t=95..115s)
chunk = peak_normalize(y2)[int(95 * sr2):int(115 * sr2)]
chunk = notch_filter(bandpass_filter(chunk, sr2), sr2, PERSISTENT_TONES["2.mp3"])
chunk_ss, noise_prof = spectral_subtraction(chunk, sr2)

fig = plt.figure(figsize=(14, 7))
gs = fig.add_gridspec(2, 2, width_ratios=[3, 1])
for row, (s, title) in enumerate([(chunk, "before"), (chunk_ss, "after spectral subtraction")]):
    ax = fig.add_subplot(gs[row, 0])
    S_db = librosa.amplitude_to_db(np.abs(librosa.stft(s, n_fft=FRAME, hop_length=HOP)), ref=np.max)
    librosa.display.specshow(S_db, sr=sr2, hop_length=HOP, x_axis="time", y_axis="hz",
                             ax=ax, cmap="magma", vmin=-80)
    ax.set_ylim(0, 4000); ax.set_title(f"2.mp3 (t=95–115 s, incl. dead air) — {title}")
axn = fig.add_subplot(gs[:, 1])
fbins = librosa.fft_frequencies(sr=sr2, n_fft=FRAME)
axn.plot(20 * np.log10(noise_prof[:, 0] + 1e-10), fbins, lw=0.8)
axn.set_ylim(0, 4000); axn.set_xlabel("noise magnitude (dB)")
axn.set_title("estimated noise profile")
fig.tight_layout(); plt.show()

## 1.5 Voice Activity Detection & silence stripping

**Method (energy-based VAD).**

1. Compute frame RMS in dB (46 ms frames, 12 ms hop).
2. Estimate the noise floor as the 10th percentile of frame energies.
3. A frame is *speech* if it exceeds `floor + 10 dB` (never lower than `max − 45 dB`).
4. **Smoothing/hysteresis:** a 7-frame median filter removes single-frame flips (a spurious blip
   doesn't become "speech", a glottal stop doesn't split a word).
5. **Padding:** the mask is dilated by 100 ms on each side so word onsets/offsets (which ramp up
   gently) are never chopped.
6. **Pause collapsing:** silences longer than 0.3 s are shortened *to* 0.3 s (not to zero — a
   completely pause-free recording sounds unnatural and is harder to transcribe).
7. Each cut is cross-faded over 10 ms to avoid clicks at the joins.

VAD runs on the *cleaned* signal: after noise removal the speech/silence contrast is much larger,
making the threshold far more reliable.

In [ ]:
def vad_mask(y, sr, margin_db=10.0, med_frames=7, pad_s=0.10):
    """Boolean per-frame speech mask via adaptive RMS thresholding."""
    db = frame_rms_db(y)
    floor = np.percentile(db, 10)
    thr = max(floor + margin_db, db.max() - 45)
    mask = db > thr
    mask = median_filter(mask, size=med_frames)
    mask = binary_dilation(mask, iterations=max(1, int(pad_s * sr / HOP)))
    return mask, thr, floor


def strip_silence(y, sr, mask, keep_pause=0.3, fade_s=0.01):
    """Collapse every silent run longer than keep_pause down to keep_pause seconds."""
    segments, i, n = [], 0, len(mask)
    while i < n:
        j = i
        while j < n and mask[j] == mask[i]:
            j += 1
        s, e = i * HOP, min(j * HOP, len(y))
        if not mask[i]:                       # silent run: keep at most keep_pause seconds
            e = min(s + int(keep_pause * sr), e)
        segments.append((s, e))
        i = j
    fade = int(fade_s * sr)
    out = []
    for s, e in segments:
        seg = y[s:e].copy()
        if len(seg) > 2 * fade:               # 10 ms fade at each cut boundary -> no clicks
            seg[:fade] *= np.linspace(0, 1, fade)
            seg[-fade:] *= np.linspace(1, 0, fade)
        out.append(seg)
    return np.concatenate(out)


# Demonstration on the cleaned 2.mp3 chunk from the previous section
mask_demo, thr_demo, floor_demo = vad_mask(chunk_ss, sr2)
db_demo = frame_rms_db(chunk_ss)
tf = np.arange(len(db_demo)) * HOP / sr2

fig, axes = plt.subplots(2, 1, figsize=(14, 5), sharex=True)
t = np.arange(len(chunk_ss)) / sr2
axes[0].plot(t[::10], chunk_ss[::10], lw=0.3)
axes[0].fill_between(tf, -1, 1, where=mask_demo, alpha=0.15, color="green", label="speech (kept)")
axes[0].legend(loc="upper right"); axes[0].set_ylim(-0.8, 0.8)
axes[0].set_title("VAD on cleaned 2.mp3 chunk — green = kept as speech")
axes[1].plot(tf, db_demo, lw=0.7)
axes[1].axhline(thr_demo, color="r", ls="--", label=f"threshold {thr_demo:.0f} dB")
axes[1].axhline(floor_demo, color="gray", ls=":", label=f"noise floor {floor_demo:.0f} dB")
axes[1].set_xlabel("time (s)"); axes[1].set_ylabel("frame RMS (dB)"); axes[1].legend()
fig.tight_layout(); plt.show()

stripped_demo = strip_silence(chunk_ss, sr2, mask_demo)
print(f"chunk: {len(chunk_ss)/sr2:.1f} s -> {len(stripped_demo)/sr2:.1f} s after silence stripping")

## 1.6 Hold-music detection & suppression

**Detection — why chroma stability?** Hold music consists of *sustained, repeating pitches*,
while speech hops between phonemes every ~100 ms. We compute the chroma vector (energy in the 12
pitch classes) per frame, and for every 5-second block measure the average correlation between
consecutive chroma frames. Speech gives low correlation (~0.3–0.7); sustained tones/music give
very high correlation (>0.8). Blocks quieter than `max − 30 dB` are ignored (silence would also
look "stable"). The longest run of stable blocks is declared the hold-music region.

**Suppression — why notch filters?** Inside the detected region, the *median* spectrum exposes
the persistent music tones (§1.1 found 2143 & 2245 Hz for `3.mp3`). These lie inside the speech
band, so the band-pass cannot help — but they are extremely narrow, so a cascade of high-Q IIR
notches removes them with minimal collateral damage. The notched signal is cross-faded (0.5 s
ramps) with the original **only inside the detected region**, so the rest of the file is not
touched at all. This is the "frequency notch-filter for the music" strategy suggested in the
handout; it removes the *tonal skeleton* of the music rather than the music entirely, which is
the honest limit of linear time-invariant filtering (a full separation would need e.g.
source-separation models).

In [ ]:
def _tcorr(c):
    """Mean correlation of consecutive chroma frames (0 = decorrelated, 1 = frozen)."""
    vals = []
    for i in range(0, c.shape[1] - 1, 3):
        a, b = c[:, i] - c[:, i].mean(), c[:, i + 1] - c[:, i + 1].mean()
        d = np.sqrt((a ** 2).sum() * (b ** 2).sum())
        vals.append((a * b).sum() / d if d > 1e-12 else 0.0)
    return float(np.mean(vals)) if vals else 0.0


def detect_music_region(y, sr, block_s=5.0, tcorr_thr=0.8, min_blocks=3):
    """Longest run of >=min_blocks 5-second blocks with highly stable chroma."""
    hop = 2048
    chroma = librosa.feature.chroma_stft(y=y, sr=sr, hop_length=hop)
    rms_db = 20 * np.log10(librosa.feature.rms(y=y, hop_length=hop)[0] + 1e-10)
    block = int(block_s * sr / hop)
    nb = chroma.shape[1] // block
    tc = np.array([_tcorr(chroma[:, b * block:(b + 1) * block]) for b in range(nb)])
    loud = np.array([rms_db[b * block:(b + 1) * block].mean() for b in range(nb)])
    stable = binary_closing(tc > tcorr_thr, structure=np.ones(3)) & (loud > rms_db.max() - 30)
    best, cur, s = None, 0, 0
    for i, m in enumerate(list(stable) + [False]):
        if m:
            if cur == 0:
                s = i
            cur += 1
        else:
            if cur >= min_blocks and (best is None or cur > best[2]):
                best = (s, i, cur)
            cur = 0
    info = (tc, loud, block_s)
    return (None if best is None else (best[0] * block_s, best[1] * block_s)), info


def suppress_music(y, sr, region, q=30.0, prom_db=10.0, max_tones=4, ramp_s=0.5):
    """Notch persistent in-band tones, cross-faded in only inside the region."""
    a, b = region
    seg = y[int(a * sr):int(b * sr)]
    tones = [f for f, p in sorted(persistent_peaks(seg, sr, fmin=400, fmax=3300,
                                                   prom_db=prom_db),
                                  key=lambda t: -t[1])[:max_tones]]
    if not tones:
        return y, []
    y_notched = notch_filter(y, sr, tones, q=q)
    w = np.zeros(len(y))
    ia, ib, ramp = int(a * sr), int(b * sr), int(ramp_s * sr)
    w[ia:ib] = 1.0
    w[max(0, ia - ramp):ia] = np.linspace(0, 1, min(ramp, ia))
    w[ib:min(len(y), ib + ramp)] = np.linspace(1, 0, min(ramp, len(y) - ib))
    return w * y_notched + (1 - w) * y, tones


# Run detection on all three files (raw, peak-normalized)
music_regions = {}
fig, axes = plt.subplots(3, 1, figsize=(14, 7), sharey=True)
for ax, name in zip(axes, AUDIO_FILES):
    y, sr = audio_raw[name]
    region, (tc, loud, bs) = detect_music_region(peak_normalize(y), sr)
    music_regions[name] = region
    tb = np.arange(len(tc)) * bs
    ax.bar(tb, tc, width=bs * 0.9, align="edge", alpha=0.7)
    ax.axhline(0.8, color="r", ls="--", lw=0.8, label="stability threshold")
    if region:
        ax.axvspan(region[0], region[1], color="orange", alpha=0.25, label="detected music")
    ax.set_ylabel("chroma stability"); ax.set_ylim(0, 1.05)
    ax.set_title(f"{name} — music region: {region}")
    ax.legend(loc="upper left", fontsize=8)
axes[-1].set_xlabel("time (s)")
fig.tight_layout(); plt.show()

In [ ]:
y3, sr3 = audio_raw["3.mp3"]
y3_pre = notch_filter(bandpass_filter(peak_normalize(y3), sr3), sr3, PERSISTENT_TONES["3.mp3"])
y3_pre, _ = adaptive_declick(y3_pre, sr3)
y3_pre, _ = spectral_subtraction(y3_pre, sr3)
region3 = music_regions["3.mp3"]
y3_supp, tones3 = suppress_music(y3_pre, sr3, region3)
print(f"3.mp3: detected music region {region3}, notched tones: "
      + ", ".join(f"{t:.0f} Hz" for t in tones3))

a, b = int(region3[0] * sr3), int(region3[1] * sr3)
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, (s, title) in zip(axes, [(y3_pre[a:b], "before"), (y3_supp[a:b], "after tone notching")]):
    S_db = librosa.amplitude_to_db(np.abs(librosa.stft(s, n_fft=4096, hop_length=1024)), ref=np.max)
    librosa.display.specshow(S_db, sr=sr3, hop_length=1024, x_axis="time", y_axis="hz",
                             ax=ax, cmap="magma", vmin=-70)
    ax.set_ylim(0, 4000); ax.set_title(f"3.mp3 hold-music region — {title}")
fig.tight_layout(); plt.show()

## 1.7 Echo analysis & cancellation

**Model.** A single acoustic reflection adds a delayed, attenuated copy:
$y[n] = x[n] + \alpha\, x[n-D]$. This is an FIR "comb" system with transfer function
$H(z) = 1 + \alpha z^{-D}$.

**Detection.** The autocorrelation of $y$ contains a peak at lag $D$ with relative height
$\approx \alpha/(1+\alpha^2)$. We search lags 40–500 ms (typical room/line echoes) with a
prominence test, so ordinary pitch periodicity (lags < 15 ms) and the slow decay of the
envelope don't produce false alarms.

**Cancellation.** If the echo parameters are known, the *exact inverse* system is the IIR filter
$H^{-1}(z) = 1/(1 + \alpha z^{-D})$, implemented recursively as
$\hat x[n] = y[n] - \alpha\, \hat x[n-D]$ (stable because $\alpha < 1$).

**Honest finding for the real recordings:** the autocorrelation of all three cleaned files shows
**no peak with prominence > 0.1** anywhere in 40–500 ms — there is no measurable discrete echo to
cancel. Rather than "cancel" a non-existent echo, we validate the complete detect-and-cancel
machinery on a **synthetic echo** injected into real call audio (delay 180 ms, α = 0.45): the
detector recovers the delay exactly and the inverse comb removes the echo to machine precision.

In [ ]:
def echo_autocorr(y, sr, max_lag_s=0.5):
    seg = y - y.mean()
    seg = seg[: min(len(seg), 60 * sr)]        # 60 s is plenty for a stable estimate
    ac = librosa.autocorrelate(seg, max_size=int(max_lag_s * sr))
    return ac / (ac[0] + 1e-12)


def detect_echo(y, sr, min_lag_s=0.04, max_lag_s=0.5, min_prominence=0.10):
    ac = echo_autocorr(y, sr, max_lag_s)
    lo = int(min_lag_s * sr)
    pk, props = find_peaks(ac[lo:], prominence=min_prominence)
    if len(pk) == 0:
        return None, ac
    best = pk[np.argmax(props["prominences"])]
    return ((lo + best) / sr, ac[lo + best]), ac


def add_echo(x, sr, delay_s, alpha):
    d = int(delay_s * sr)
    return lfilter(np.concatenate(([1.0], np.zeros(d - 1), [alpha])), [1.0], x)


def cancel_echo(y, sr, delay_s, alpha):
    d = int(delay_s * sr)
    return lfilter([1.0], np.concatenate(([1.0], np.zeros(d - 1), [alpha])), y)


# --- Real files: search for echo ---
fig, axes = plt.subplots(1, 3, figsize=(14, 3.5), sharey=True)
for ax, name in zip(axes, AUDIO_FILES):
    y, sr = audio_raw[name]
    yc = bandpass_filter(peak_normalize(y), sr)
    hit, ac = detect_echo(yc, sr)
    lags = np.arange(len(ac)) / sr * 1000
    ax.plot(lags, ac, lw=0.6)
    ax.set_xlim(40, 500); ax.set_ylim(-0.2, 0.5)
    ax.axhline(0.1, color="r", ls="--", lw=0.8)
    ax.set_title(f"{name}: {'no echo peak' if hit is None else f'{hit[0]*1000:.0f} ms'}")
    ax.set_xlabel("lag (ms)")
axes[0].set_ylabel("normalized autocorrelation")
fig.suptitle("Echo search on the real recordings (red line = detection threshold)")
fig.tight_layout(); plt.show()

In [ ]:
# --- Synthetic validation of the detect-and-cancel machinery ---
x_clean = bandpass_filter(peak_normalize(audio_raw["2.mp3"][0]), 44100)[int(120 * 44100):int(140 * 44100)]
sr_demo = 44100
DELAY, ALPHA = 0.180, 0.45

y_echo = add_echo(x_clean, sr_demo, DELAY, ALPHA)
hit, ac_echo = detect_echo(y_echo, sr_demo)
print(f"injected echo:  delay = {DELAY*1000:.0f} ms, alpha = {ALPHA}")
print(f"detected echo:  delay = {hit[0]*1000:.0f} ms, autocorr height = {hit[1]:.3f}")

x_rec = cancel_echo(y_echo, sr_demo, hit[0], ALPHA)
hit2, ac_rec = detect_echo(x_rec, sr_demo)
print(f"after inverse comb: residual |x_rec - x_clean|_max = {np.max(np.abs(x_rec - x_clean)):.2e}, "
      f"echo peak: {'gone' if hit2 is None else hit2}")

fig, axes = plt.subplots(1, 2, figsize=(14, 3.5), sharey=True)
for ax, (ac, title) in zip(axes, [(ac_echo, "echoed signal"), (ac_rec, "after cancellation")]):
    lags = np.arange(len(ac)) / sr_demo * 1000
    ax.plot(lags, ac, lw=0.6); ax.set_xlim(40, 500); ax.set_ylim(-0.2, 0.6)
    ax.axhline(0.1, color="r", ls="--", lw=0.8)
    ax.set_title(f"autocorrelation — {title}"); ax.set_xlabel("lag (ms)")
fig.tight_layout(); plt.show()

## 1.8 The complete pipeline, applied to all three recordings

`enhance_file` chains all stages in the order motivated in §1.2 and records metrics after each
stage. The quality metrics:

* **SNR estimate** — mean speech-frame energy minus the noise-floor (10th percentile) energy, in
  dB. It is only meaningful *before* silence stripping (the floor estimate needs silence to
  measure), so the final table row also reports the pre-strip SNR: recomputing it on the
  stripped signal would make the 10th percentile ride on retained speech pauses and show an
  artificial dip that has nothing to do with quality.
* **silence fraction** — frames more than 40 dB below the loudest frame.
* **duration** — shows how much dead air VAD removed.
* **RMS / peak** — shows loudness normalization worked and nothing clipped.

For listening, the notebook embeds a 15 s before/after excerpt per file (full enhanced files are
written to `outputs/enhanced_*.wav`).

In [ ]:
import IPython.display as ipd


def snr_estimate(y, margin_db=10.0):
    db = frame_rms_db(y)
    floor = np.percentile(db, 10)
    speech = db[db > floor + margin_db]
    return float(speech.mean() - floor) if len(speech) else np.nan


def audio_metrics(y, sr):
    db = frame_rms_db(y)
    return {
        "dur": len(y) / sr,
        "rms_db": 20 * np.log10(np.sqrt(np.mean(y ** 2)) + 1e-12),
        "peak": float(np.max(np.abs(y))),
        "snr": snr_estimate(y),
        "silence": float(np.mean(db < db.max() - 40)),
    }


def enhance_file(name, verbose=True):
    y_raw, sr = audio_raw[name]
    log = []

    y = peak_normalize(y_raw)
    log.append(("raw (peak-normalized)", audio_metrics(y, sr)))

    tones_bp = PERSISTENT_TONES[name]
    y = notch_filter(bandpass_filter(y, sr), sr, tones_bp)
    log.append((f"band-pass + tone notches {[f'{t:.0f}' for t in tones_bp]}",
                audio_metrics(y, sr)))

    y, n_clicks = adaptive_declick(y, sr)
    log.append((f"de-click ({n_clicks} samples)", audio_metrics(y, sr)))

    y, _ = spectral_subtraction(y, sr)
    log.append(("spectral subtraction", audio_metrics(y, sr)))

    region = music_regions[name]
    tones = []
    if region is not None:
        y, tones = suppress_music(y, sr, region)
        log.append((f"music notches {[f'{t:.0f}' for t in tones]} Hz", audio_metrics(y, sr)))

    echo_hit, _ = detect_echo(y, sr)

    mask, thr, floor = vad_mask(y, sr)
    y_stripped = strip_silence(y, sr, mask)
    y_final = rms_normalize(y_stripped)
    m_final = audio_metrics(y_final, sr)
    m_final["snr"] = snr_estimate(y)   # SNR is measured pre-strip (see the metric note above)
    log.append(("VAD-strip + RMS normalize", m_final))

    if verbose:
        print(f"\n================ {name} ================")
        print(f"music region: {region},  echo: "
              f"{'none detected' if echo_hit is None else f'{echo_hit[0]*1000:.0f} ms ({echo_hit[1]:.2f})'}")
        rows = [[stage, f"{m['dur']:.1f}", f"{m['rms_db']:.1f}", f"{m['peak']:.2f}",
                 f"{m['snr']:.1f}", f"{100*m['silence']:.0f}%"] for stage, m in log]
        show_table(["stage", "dur (s)", "RMS dBFS", "peak", "SNR est (dB)", "silence"], rows)
    return y_final, sr, log


enhanced = {}
for name in AUDIO_FILES:
    y_final, sr, log = enhance_file(name)
    enhanced[name] = (y_final, sr)
    sf.write(OUT_DIR / f"enhanced_{name.split('.')[0]}.wav", y_final, sr)
print("\nenhanced files written to", OUT_DIR.resolve())

In [ ]:
# Before/after spectrograms of the full files
for name in AUDIO_FILES:
    y_raw, sr = audio_raw[name]
    y_fin, _ = enhanced[name]
    fig, axes = plt.subplots(2, 1, figsize=(14, 6))
    for ax, (s, title) in zip(axes, [(peak_normalize(y_raw), "original"),
                                     (y_fin, "enhanced (note the shorter time axis)")]):
        S_db = librosa.amplitude_to_db(np.abs(librosa.stft(s, n_fft=FRAME, hop_length=HOP)),
                                       ref=np.max)
        librosa.display.specshow(S_db, sr=sr, hop_length=HOP, x_axis="time", y_axis="hz",
                                 ax=ax, cmap="magma", vmin=-80)
        ax.set_ylim(0, 8000); ax.set_title(f"{name} — {title}")
    fig.tight_layout(); plt.show()

In [ ]:
# A/B listening excerpts (15 s), embedded at 22.05 kHz to keep the notebook small.
# Full-length enhanced audio: outputs/enhanced_*.wav
def excerpt(y, sr, t0, dur=15):
    e = y[int(t0 * sr):int((t0 + dur) * sr)]
    return librosa.resample(e, orig_sr=sr, target_sr=22050), 22050

for name, t0 in [("2.mp3", 105), ("3.mp3", 85), ("4.mp3", 20)]:
    y_raw, sr = audio_raw[name]
    y_fin, _ = enhanced[name]
    print(f"--- {name}: original (t={t0}s) vs enhanced (same neighbourhood) ---")
    ea, sra = excerpt(peak_normalize(y_raw), sr, t0)
    eb, srb = excerpt(y_fin, sr, min(t0, max(0, len(y_fin) / sr - 15)))
    ipd.display(ipd.Audio(ea, rate=sra))
    ipd.display(ipd.Audio(eb, rate=srb))

## 1.9 Comparative analysis & report

| Artifact | Technique | Difficulty & outcome |
|---|---|---|
| Out-of-band noise (hiss above 3.4 kHz, rumble + mains hum at 48/150/250 Hz) | Butterworth band-pass, zero-phase | **Easiest.** The artifact and the signal do not overlap in frequency, so a fixed LTI filter removes it completely with zero speech damage. |
| In-band persistent tones (e.g. the 2966 Hz line in `4.mp3`) | Median-spectrum detection + high-Q IIR notches | **Easy.** Perfectly narrowband and stationary → auto-detected from the median spectrum and removed with a surgical notch. |
| Stationary hiss inside the speech band | Spectral subtraction (α=2, β=0.05) | **Medium.** Solid stage gain (e.g. `3.mp3` +9.7 dB for this stage, 29.4 → 39.1 dB; the cumulative gain from raw is ~27 dB), but aggressive settings introduce audible "musical noise"; α/β are a quality-vs-artifact trade-off. |
| Impulsive crackle / clicks | Adaptive local limiter | **Medium.** Detection is easy; making it *never* touch loud speech required a local (rolling-RMS) threshold instead of a global one. |
| Dead air / long silences | Energy VAD + pause collapsing | **Easy after cleaning.** On the raw file the threshold is ambiguous; after noise removal the speech/noise contrast grows by >20 dB and a simple threshold works. Faint hold-beeps inside the dead air are (correctly) kept as non-silence. |
| Hold music | Chroma-stability detection + region-limited tone notching | **Hard.** Detection worked cleanly (only `3.mp3`, t≈90–135 s). Removal is fundamentally limited: only the sustained tonal lines can be notched by LTI filtering; the broadband part of the music overlaps speech in time *and* frequency, and true separation would need source-separation models outside this course's toolbox. |
| Echo | Autocorrelation detection + inverse comb | **No echo present.** The detector (validated on synthetic echo: exact delay recovery, cancellation to machine precision) finds no autocorrelation peak with prominence > 0.1 in any file — applying an "echo canceller" anyway would only distort the audio, so we report the negative result honestly. |
| Loudness differences (between files & speakers) | RMS normalization to −20 dBFS | **Easiest.** `3.mp3` lands exactly at −20 dBFS; `2.mp3`/`4.mp3` settle ~2.5 dB lower because the no-clipping peak guard (peak ≤ 0.99) limits the gain — a deliberate trade of loudness for zero distortion. |

**Overall:** the frequency-domain artifacts with fixed structure (band-limit, hum, tones) were
the easiest — they are exactly what LTI filters are built for. The hardest artifacts are the
ones that *overlap speech in both time and frequency* (hold music, non-stationary noise), where
any removal necessarily trades off speech quality; there the pragmatic engineering answer is
partial suppression plus honest reporting.

---
# Problem 2 — Automated Wellbore Fracture Segmentation

## Background theory (used throughout Problem 2)

An image is a **2-D discrete signal** $I[r, c]$; everything from Problem 1 has a 2-D analogue.
The tools we use:

* **Grayscale & histograms.** Fractures are *dark* structures on a *bright* rock background, so
  a single intensity channel carries all the information.
* **CLAHE (Contrast-Limited Adaptive Histogram Equalization).** Ordinary histogram equalization
  stretches contrast globally — but borehole scans have uneven illumination, so a global
  stretch under/over-shoots locally. CLAHE equalizes each tile (8×8 grid) separately with a clip
  limit that stops noise from being amplified too strongly.
* **Median filtering** replaces each pixel by the median of its neighbourhood. Unlike Gaussian
  blur it *removes* structures that occupy less than half its window entirely (it is a rank
  filter, not an averaging filter) while keeping edges of larger structures sharp — ideal both
  for speckle and, with a **1×11 horizontal kernel**, for the thin *vertical striping* artifact
  (a vertical stripe is only 1–3 px wide horizontally — well under the ~5 px half-window — so a
  horizontal median annihilates it, while wide fracture traces survive).
* **Morphology.** With a binary image and a *structuring element* (SE):
  **erosion** = local minimum (shrinks white), **dilation** = local maximum (grows white),
  **opening** = erode→dilate (removes white structures *thinner than the SE*),
  **closing** = dilate→erode (fills gaps *narrower than the SE*).
  On grayscale images, **black-hat** = closing(I) − I responds exactly to *dark structures
  narrower than the SE* — a perfect matched detector for dark fractures up to ~31 px thick.
* **Ridge vs edge detection.** Sobel/Canny find *edges* (intensity discontinuities) — a thick
  dark fracture yields **two** parallel edges and a hollow interior. Ridge/vessel filters
  (Frangi) or black-hat respond to the *entire dark ribbon*, which is what we must keep, so they
  match the problem better (§2.3 shows this side by side).
* **Hysteresis thresholding** (as in Canny): keep every pixel above a *strong* threshold, plus
  any pixel above a *weak* threshold that is **connected** to a strong one. Faint continuations
  of strong fractures survive; isolated faint noise does not.
* **Connected components** label each maximal set of touching foreground pixels, giving per-
  component statistics (bounding box, area) — the tool for the final "does it cross the image?"
  continuity test.

## 2.1 EDA — what is in these images?

Note three things in the raw scans:

1. **Primary fractures**: thick, dark, continuous sinusoidal traces crossing the full width
   (a planar fracture intersecting a cylindrical borehole unwraps to a sinusoid).
2. **Micro-cracks**: thin, fragmented dark filaments, often clustered.
3. **Vertical striping**: a dense field of 1–3 px wide vertical dark streaks (an imaging
   artifact of the logging tool), strongest in the left half of `Well1.jpg` — visible as
   high-frequency jitter in the column-mean brightness profile.

In [ ]:
IMG_FILES = ["Well1.jpg", "well2.png"]
images = {n: cv2.imread(str(DATA_DIR / n)) for n in IMG_FILES}
grays = {n: cv2.cvtColor(im, cv2.COLOR_BGR2GRAY) for n, im in images.items()}

for name in IMG_FILES:
    g = grays[name]
    H, W = g.shape
    fig = plt.figure(figsize=(14, 6))
    gs = fig.add_gridspec(3, 2, height_ratios=[3, 1.2, 1.2])
    ax0 = fig.add_subplot(gs[0, :])
    ax0.imshow(g, cmap="gray", aspect="auto")
    ax0.set_title(f"{name} — {W}x{H} px, unwrapped borehole scan"); ax0.axis("off")

    ax1 = fig.add_subplot(gs[1, :])
    ax1.plot(g.mean(axis=0), lw=0.5)
    ax1.set_xlim(0, W); ax1.set_ylabel("col mean")
    ax1.set_title("column-mean brightness (high-frequency jitter = vertical striping)")

    ax2 = fig.add_subplot(gs[2, 0])
    ax2.hist(g.ravel(), bins=64, color="gray")
    ax2.set_title("intensity histogram")
    ax3 = fig.add_subplot(gs[2, 1])
    crop = g[:, int(0.55 * W):int(0.75 * W)]
    ax3.imshow(crop, cmap="gray", aspect="auto")
    ax3.set_title("zoom: thick primary trace vs thin micro-cracks"); ax3.axis("off")
    fig.tight_layout(); plt.show()

## 2.2 Pre-processing & filtering

Three steps, each targeting one nuisance:

1. **CLAHE** (clip 1.5, 8×8 tiles) — evens out illumination and boosts faint fracture contrast.
   The clip limit is kept *low* because an aggressive CLAHE amplifies the striping field.
2. **Horizontal 1×11 median** — destriping: a median filter erases structures that fill less
   than half its window, i.e. anything narrower than ~5 px horizontally — exactly the 1–3 px
   vertical stripes — while fracture ribbons, which are locally much wider, survive.
3. **9×9 median blur** — removes speckle and, crucially, *erases most micro-cracks outright*
   (they are thinner than half the kernel), which is thickness-selection acting already in the
   intensity domain.

In [ ]:
def preprocess(gray, clahe_clip=1.5, hmed=11, med=9):
    eq = cv2.createCLAHE(clipLimit=clahe_clip, tileGridSize=(8, 8)).apply(gray)
    destriped = median_filter(eq, size=(1, hmed))
    smoothed = cv2.medianBlur(destriped, med)
    return eq, destriped, smoothed


pre = {}
for name in IMG_FILES:
    eq, destriped, smoothed = preprocess(grays[name])
    pre[name] = smoothed
    panels = [(grays[name], "grayscale"), (eq, "CLAHE (clip 1.5)"),
              (destriped, "after 1x11 horizontal median (destriped)"),
              (smoothed, "after 9x9 median (micro-cracks & speckle suppressed)")]
    fig, axes = plt.subplots(4, 1, figsize=(14, 9))
    for ax, (im, title) in zip(axes, panels):
        ax.imshow(im, cmap="gray", aspect="auto")
        ax.set_title(f"{name} — {title}", fontsize=9); ax.axis("off")
    fig.tight_layout(); plt.show()

## 2.3 Edge & ridge detection — choosing the detector

Side-by-side on a crop of `Well1.jpg`: Sobel gradient magnitude, Canny, Frangi vesselness, and
morphological **black-hat** (SE 31×31).

* **Sobel/Canny** outline every fracture with double edges and respond strongly to residual
  striping — they detect *boundaries*, not *ribbons*.
* **Frangi** responds to ridge-like structures but, at the scales needed for thick fractures,
  also lights up the striping field and yields hollow/split responses on the thickest traces.
* **Black-hat** returns the full dark ribbon of any fracture thinner than its SE with a clean,
  proportional response — this is what the rest of the pipeline uses. We then binarize its
  response with **hysteresis** (strong = 95th percentile, weak = 83rd), so faint fracture
  segments survive only where they connect to strong ones.

In rubric terms: Sobel, Canny and Frangi are each implemented and evaluated below; the deployed
detector is black-hat (itself a morphological *ridge* detector) combined with Canny's hysteresis
mechanism — chosen from this comparison on evidence, not by default.

In [ ]:
crop = pre["Well1.jpg"][:, 1500:2400]
sobel = np.hypot(cv2.Sobel(crop, cv2.CV_64F, 1, 0, ksize=3),
                 cv2.Sobel(crop, cv2.CV_64F, 0, 1, ksize=3))
canny = cv2.Canny(crop, 50, 150)
fr = frangi(1.0 - crop.astype(float) / 255.0, sigmas=(3, 5, 7, 9), black_ridges=False)
bh_crop = cv2.morphologyEx(crop, cv2.MORPH_BLACKHAT,
                           cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (31, 31)))

panels = [(crop, "pre-processed crop"), (sobel, "Sobel magnitude"), (canny, "Canny"),
          (np.power(fr / fr.max(), 0.4), "Frangi vesselness (gamma for display)"),
          (bh_crop, "black-hat 31x31  <-- chosen")]
fig, axes = plt.subplots(5, 1, figsize=(13, 11))
for ax, (im, title) in zip(axes, panels):
    ax.imshow(im, cmap="gray", aspect="auto")
    ax.set_title(title, fontsize=9); ax.axis("off")
fig.tight_layout(); plt.show()

In [ ]:
def detect_ridges(smoothed, bh_size=31, p_weak=83, p_strong=95):
    """Black-hat response + hysteresis binarization."""
    bh = cv2.morphologyEx(smoothed, cv2.MORPH_BLACKHAT,
                          cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (bh_size, bh_size)))
    lo, hi = np.percentile(bh, p_weak), np.percentile(bh, p_strong)
    binary = (apply_hysteresis_threshold(bh, lo, hi) * 255).astype(np.uint8)
    return bh, binary


ridge = {}
for name in IMG_FILES:
    bh, binary = detect_ridges(pre[name])
    ridge[name] = binary
    fig, axes = plt.subplots(2, 1, figsize=(14, 5))
    axes[0].imshow(bh, cmap="gray", aspect="auto")
    axes[0].set_title(f"{name} — black-hat response"); axes[0].axis("off")
    axes[1].imshow(binary, cmap="gray", aspect="auto")
    axes[1].set_title("hysteresis binarization (weak p83 / strong p95)"); axes[1].axis("off")
    fig.tight_layout(); plt.show()

## 2.4 Morphological selection — thickness & shape

Micro-cracks that survived detection are eliminated by two complementary criteria:

1. **Thickness — opening with a 5×5 elliptical SE**: any white structure thinner than ~5 px is
   erased entirely; thick primary ribbons survive essentially intact (the dilation half of the
   opening restores their body after the erosion).
2. **Shape — elongation filter**: fracture fragments are *curvilinear* (long and thin as
   regions), residual noise blobs are *compact*. Using connected-component ellipse fits we keep
   fragments with `major/minor axis ratio >= 3` (or very large area, which covers washout zones),
   and drop small compact blobs.
3. **Closing with a 15×5 SE** (wider than tall, matching the near-horizontal fracture
   orientation) re-connects small gaps left by the opening.

In [ ]:
def morph_select(binary, open_size=5, min_elong=3.0, min_frag_area=350, big_area=4000,
                 close_w=15, close_h=5):
    opened = cv2.morphologyEx(binary, cv2.MORPH_OPEN,
                              cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (open_size, open_size)))
    lab = label(opened > 0, connectivity=2)
    shaped = np.zeros_like(opened)
    kept = dropped = 0
    for r in regionprops(lab):
        elong = r.axis_major_length / (r.axis_minor_length + 1e-6)
        if r.area >= big_area or (r.area >= min_frag_area and elong >= min_elong):
            shaped[lab == r.label] = 255
            kept += 1
        else:
            dropped += 1
    closed = cv2.morphologyEx(shaped, cv2.MORPH_CLOSE,
                              cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (close_w, close_h)))
    return opened, shaped, closed, kept, dropped


morphed = {}
for name in IMG_FILES:
    opened, shaped, closed, kept, dropped = morph_select(ridge[name])
    morphed[name] = closed
    print(f"{name}: shape filter kept {kept} fragments, dropped {dropped}")
    panels = [(opened, "after 5x5 opening (thin cracks gone)"),
              (shaped, f"after elongation filter (dropped {dropped} compact blobs)"),
              (closed, "after 15x5 closing (gaps bridged)")]
    fig, axes = plt.subplots(3, 1, figsize=(14, 7))
    for ax, (im, title) in zip(axes, panels):
        ax.imshow(im, cmap="gray", aspect="auto")
        ax.set_title(f"{name} — {title}", fontsize=9); ax.axis("off")
    fig.tight_layout(); plt.show()

## 2.5 Continuity verification

**Goal:** keep only fracture networks that genuinely traverse the image horizontally.

**Linking trick.** A primary fracture can be interrupted by washed-out zones (e.g. the pale
band around x≈700–1300 in `Well1.jpg`, where the tool recorded almost no contrast — visibly a
*data* gap, not a geological one). We therefore build a **linking mask**: the cleaned mask
dilated with a wide 101×11 SE. Fragments that lie along the same trace merge in the linking
mask; the linking mask is used **only for the connectivity test**, while the output keeps the
crisp un-dilated shapes.

**Criterion.** Connected components of the linking mask are kept if their bounding-box width is
at least **25 % of the image width**. The margin is comfortable: in both graded images the true
networks span 30–100 % of the width while the largest noise cluster spans < 14 % (table below).
We report each component's horizontal extent and the **column coverage** of the final mask
(fraction of image columns crossed by at least one kept fracture) as the quantitative
verification that the selected fractures cross the image.

In [ ]:
def continuity_filter(closed, link_w=101, link_h=11, min_width_frac=0.25):
    H, W = closed.shape
    link = cv2.dilate(closed, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (link_w, link_h)))
    n, labels_img, stats, _ = cv2.connectedComponentsWithStats(link, connectivity=8)
    rows, keep_ids = [], []
    for i in range(1, n):
        wfrac = stats[i, cv2.CC_STAT_WIDTH] / W
        kept = wfrac >= min_width_frac
        if kept:
            keep_ids.append(i)
        rows.append([i, f"{wfrac:.2f}", stats[i, cv2.CC_STAT_AREA], "KEEP" if kept else "drop"])
    final = np.where(np.isin(labels_img, keep_ids), closed, 0).astype(np.uint8)
    coverage = float(np.mean((final > 0).any(axis=0)))
    rows.sort(key=lambda r: -float(r[1]))
    return final, link, rows, coverage


finals = {}
for name in IMG_FILES:
    final, link, rows, coverage = continuity_filter(morphed[name])
    finals[name] = final
    show_table(["component", "width/W", "area (px)", "verdict"], rows[:10],
               f"\n{name} — linking-mask components (top 10 by width):")
    print(f"column coverage of kept networks: {coverage*100:.0f}%")

    overlay = images[name].copy()
    overlay[final > 0] = (0, 0, 255)
    panels = [(link, "linking mask (continuity test only)"),
              (final, "FINAL primary-fracture mask"),
              (cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB), "overlay on original")]
    fig, axes = plt.subplots(3, 1, figsize=(14, 7))
    for ax, (im, title) in zip(axes, panels):
        ax.imshow(im, cmap="gray" if im.ndim == 2 else None, aspect="auto")
        ax.set_title(f"{name} — {title}", fontsize=9); ax.axis("off")
    fig.tight_layout(); plt.show()

    cv2.imwrite(str(OUT_DIR / f"fractures_{name.split('.')[0]}_mask.png"), final)
    cv2.imwrite(str(OUT_DIR / f"fractures_{name.split('.')[0]}_overlay.png"), overlay)
print("masks and overlays written to", OUT_DIR.resolve())

## 2.6 Generalization check on the extra scans

The full pipeline (identical parameters, no per-image tuning) applied to the 8 additional scans
in `More Well images/`.

In [ ]:
def segment_fractures(gray):
    """Full Problem-2 pipeline: returns final mask + stats."""
    _, _, smoothed = preprocess(gray)
    _, binary = detect_ridges(smoothed)
    _, _, closed, _, _ = morph_select(binary)
    final, _, rows, coverage = continuity_filter(closed)
    return final, rows, coverage


extra = sorted((DATA_DIR / "More Well images").glob("*.png"))
fig, axes = plt.subplots(len(extra), 1, figsize=(14, 2.1 * len(extra)))
for ax, p in zip(axes, extra):
    img = cv2.imread(str(p))
    final, rows, coverage = segment_fractures(cv2.cvtColor(img, cv2.COLOR_BGR2GRAY))
    kept = sum(1 for r in rows if r[3] == "KEEP")
    overlay = img.copy()
    overlay[final > 0] = (0, 0, 255)
    ax.imshow(cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB), aspect="auto")
    ax.set_title(f"{p.name}: {kept} primary networks kept, column coverage {coverage*100:.0f}%",
                 fontsize=9)
    ax.axis("off")
fig.tight_layout(); plt.show()

## 2.7 Discussion & limitations

**What worked.**
* The **black-hat + hysteresis** detector proved much better matched to "dark thick ribbons"
  than edge detectors (§2.3) and is nearly immune to the striping after the 1×11 horizontal
  median destriping.
* The two-criterion micro-crack removal (**thickness** via opening, **shape** via elongation)
  eliminated the dense micro-crack clusters of `Well1.jpg` without breaking primary traces.
* The **linking-mask continuity test** is robust to genuine interruptions: in `Well1.jpg` the
  pale washed-out band around x ≈ 700–1300 carries no recorded contrast, so the fracture
  networks to its left and right are kept as two separate (still image-crossing) systems.
* With identical parameters, all 8 extra scans yield sensible primary traces — the pipeline
  generalizes.

**Limitations (honest).**
* A few compact noise blobs inside `Well1.jpg`'s striping field survive because they touch a
  kept network in the linking mask.
* The wide diffuse horizontal band in `Well1.jpg` (y ≈ 70–170) is *not* detected: at ~100 px it
  is thicker than the 31-px black-hat SE. It is arguably a bedding feature rather than a
  fracture; detecting it would need a second, larger analysis scale.
* Percentile-based thresholds assume a roughly constant fraction of fracture pixels per image;
  a scan that is almost fracture-free would have its noise stretched to fill the percentile
  budget (partially mitigated by the shape and continuity filters).
* The 25 % width criterion is a heuristic; a physics-aware alternative would fit sinusoids
  (borehole geometry) to the detected traces.

## AI-usage note (per the course's Generative-AI policy)

AI assistance (Anthropic Claude Fable 5) was used while developing this homework — for brainstorming the
pipeline structure, debugging, and drafting code and explanations. Representative prompts:
"design an audio-enhancement pipeline (band-pass, spectral subtraction, VAD) for degraded
call-center recordings and explain the DSP theory", "detect thick continuous fractures in an
unwrapped borehole image with OpenCV morphology and connected components". All generated code
was reviewed, tuned against the actual data (all thresholds/parameters were chosen from the EDA
measurements shown above), and the underlying signal-processing theory for every stage is
explained in the markdown cells of this notebook.